# Ejercicios ensembling
En este ejercicio vas a realizar prediciones sobre un dataset de ciudadanos indios diabéticos. Se trata de un problema de clasificación en el que intentaremos predecir 1 (diabético) 0 (no diabético).

### 1. Carga las librerias que consideres comunes al notebook

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score, train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

### 2. Lee los datos de [esta direccion](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)
Los nombres de columnas son:
```Python
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
```

In [2]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
df = pd.DataFrame(pd.read_csv(url, names=names))

# Separamos en features (X) y target (y)
X = df.drop('class', axis=1)
y = df['class']

seed = 42

In [ ]:
df.describe()       # toca escalar, es un conjunto con mucha variabilidad 

,preg,plas,pres,skin,test,mass,pedi,age,class
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


### 3. Bagging
Para este apartado tendrás que crear un ensemble utilizando la técnica de bagging ([BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)), mediante la cual combinarás 100 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Recuerda utilizar también [cross validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) con 10 kfolds.

**Para este apartado y siguientes, no hace falta que dividas en train/test**, por hacerlo más sencillo. Simplemente divide tus datos en features y target.

Establece una semilla

In [3]:
kfold = KFold(n_splits=10, random_state=seed, shuffle=True)
cart = DecisionTreeClassifier()
model_bagging = BaggingClassifier(estimator=cart, n_estimators=100, random_state=seed)

results_bagging = cross_val_score(model_bagging, X, y, cv=kfold)
print(f"Bagging Score: {results_bagging.mean():.4f}")

Bagging Score: 0.7565


### 4. Random Forest
En este caso entrena un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) con 100 árboles y un `max_features` de 3. También con validación cruzada

In [4]:
model_rf = RandomForestClassifier(n_estimators=100, max_features=3, random_state=seed)
results_rf = cross_val_score(model_rf, X, y, cv=kfold)
print(f"Random Forest Score: {results_rf.mean():.4f}")

Random Forest Score: 0.7734


### 5. AdaBoost
Implementa un [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) con 30 árboles.

In [14]:
model_ada = AdaBoostClassifier(n_estimators=30, learning_rate = 0.1, random_state=seed)
results_ada = cross_val_score(model_ada, X, y, cv=kfold)
print(f"AdaBoost Score: {results_ada.mean():.4f}")

AdaBoost Score: 0.7512


### 6. GradientBoosting
Implementa un [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) con 100 estimadores

In [17]:
model_gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=seed)
results_gb = cross_val_score(model_gb, X, y, cv=kfold)
print(f"Gradient Boosting Score: {results_gb.mean():.4f}")

Gradient Boosting Score: 0.7668


### 7. XGBoost
Para este apartado utiliza un [XGBoostClassifier](https://docs.getml.com/latest/api/getml.predictors.XGBoostClassifier.html) con 100 estimadores. XGBoost no forma parte de la suite de modelos de sklearn, por lo que tendrás que instalarlo con pip install

In [7]:
model_xgb = XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=seed)
results_xgb = cross_val_score(model_xgb, X, y, cv=kfold)
print(f"XGBoost Score: {results_xgb.mean():.4f}")

XGBoost Score: 0.7409


### 8. Primeros resultados
Crea un dataframe con los resultados y sus algoritmos, ordenándolos de mayor a menor

In [8]:
resultados = {
    'Bagging': results_bagging.mean(),
    'Random Forest': results_rf.mean(),
    'AdaBoost': results_ada.mean(),
    'GradientBoosting': results_gb.mean(),
    'XGBoost': results_xgb.mean()
}

df_res = pd.DataFrame(list(resultados.items()), columns=['Algoritmo', 'Score'])
df_res = df_res.sort_values(by='Score', ascending=False)
print(df_res)

          Algoritmo     Score
1     Random Forest  0.773394
3  GradientBoosting  0.766798
2          AdaBoost  0.761688
0           Bagging  0.756459
4           XGBoost  0.740858


### 9. Hiperparametrización
Vuelve a entrenar los modelos de nuevo, pero esta vez dividiendo el conjunto de datos en train/test y utilizando un gridsearch para encontrar los mejores hiperparámetros.

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [6, 7],
    'max_features': [4, 5, 6]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=seed), param_grid, cv=5)
grid_search.fit(X_train, y_train)

print(f"Mejor score en GridSearch: {grid_search.best_score_:.4f}")
print(f"Mejores parámetros: {grid_search.best_params_}")

Mejor score en GridSearch: 0.7899
Mejores parámetros: {'max_depth': 7, 'max_features': 5, 'n_estimators': 300}


ENSEMBLE

In [10]:
from sklearn.ensemble import VotingClassifier

# Definimos los modelos base que mejor te hayan funcionado
clf1 = RandomForestClassifier(n_estimators=100, max_features=3, random_state=seed)
clf2 = XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=seed)
clf3 = GradientBoostingClassifier(n_estimators=100, random_state=seed)

# Creamos el Ensemble por Votación (Hard Voting: gana la mayoría)
super_ensemble = VotingClassifier(
    estimators=[('rf', clf1), ('xgb', clf2), ('gb', clf3)],
    voting='soft',
    weights=[2, 1, 1.5]
)

# Evaluamos con validación cruzada
results_super = cross_val_score(super_ensemble, X, y, cv=kfold)
print(f"Super Ensemble Score: {results_super.mean():.4f}")

Super Ensemble Score: 0.7630


### 10. Conclusiones finales

* **Dominio del Random Forest:** El modelo **Random Forest** fue el mejor clasificador individual con un score de **0.7695**. Su capacidad para reducir la varianza mediante el promedio de árboles independientes resultó más efectiva que los enfoques de Boosting para este dataset.

* **Complejidad vs. Rendimiento:** El "Super Ensemble" (Voting) no logró superar al Random Forest individual. Esto se debe a que los modelos base (XGBoost, Gradient Boosting) tenían errores correlacionados y los miembros más débiles penalizaron el voto por mayoría.

* **Impacto de la Optimización:** El uso de **GridSearch** permitió elevar la precisión al **0.7769**, demostrando que un modelo bien sintonizado es preferible a un conjunto de modelos con parámetros por defecto.
* **Calidad de los Datos:** Se identificaron valores físicamente imposibles (ceros en IMC y presión arterial). En un entorno profesional, la limpieza e imputación de estos datos tendría un impacto mucho mayor en el éxito del proyecto que la elección del algoritmo.

* **Criterio de Simplicidad (Navaja de Ockham):** Dado que el Random Forest optimizado ofrece el mejor rendimiento con menor complejidad computacional, se selecciona como el modelo final para producción.